# 🔊 Kokoro Text-to-Speech — Google Colab Demo

This notebook runs the **[docker-kokoro](https://github.com/hwdsl2/docker-kokoro)** API server directly in Colab (without Docker).

The server provides an **OpenAI-compatible** `POST /v1/audio/speech` endpoint powered by [Kokoro TTS](https://huggingface.co/hexgrad/Kokoro-82M).

**Quick start:** Select **Runtime → Run all** to run the full demo automatically using a built-in sample text.  
**Runtime recommendation:** GPU is used automatically when available. CPU works too.

## Step 1 — Install dependencies

In [ ]:
# Core server dependencies
!pip install -q "kokoro>=0.9" soundfile fastapi "uvicorn[standard]"
print('✅ Installation complete.')

## Step 2 — Download the API server script

In [ ]:
import urllib.request

API_SERVER_URL = "https://raw.githubusercontent.com/hwdsl2/docker-kokoro/main/api_server.py"
urllib.request.urlretrieve(API_SERVER_URL, "/content/api_server.py")
print("✅ Downloaded api_server.py")

## Step 3 — Configure voice & runtime settings

Edit the variables below to choose a different voice or speech speed.

**Selected Kokoro voices:**

| Voice ID | Description |
|---|---|
| `af_heart` | American female — warm, natural **(default)** |
| `af_bella` | American female — expressive |
| `af_nova` | American female — clear |
| `am_adam` | American male — deep |
| `am_michael` | American male — clear |
| `bf_emma` | British female — clear, professional |
| `bm_george` | British male — authoritative |

OpenAI voice aliases (`alloy`, `echo`, `fable`, `onyx`, `nova`, `shimmer`, `ash`, `coral`, `sage`, `verse`) are also accepted.  
Run `GET /v1/voices` to see all available voices.

In [ ]:
import os, subprocess

# ── Voice & speech settings ──────────────────────────────────────────────────
KOKORO_VOICE = "af_heart"  # Kokoro voice ID or OpenAI alias (see table above)
KOKORO_SPEED = "1.0"       # Speech speed: 0.25 (slowest) to 4.0 (fastest)

# ── Server settings ──────────────────────────────────────────────────────────
KOKORO_PORT    = "8881"
KOKORO_API_KEY = ""        # optional Bearer token; leave empty for open access

# Restrict which language pipeline to load at startup based on voice prefix:
# 'a' for American English voices (af_* / am_*), 'b' for British (bf_* / bm_*).
# Loading only the needed pipeline saves time and memory.
# Leave empty to load both (required when switching between American and British voices).
KOKORO_LANG_CODE = KOKORO_VOICE[0] if len(KOKORO_VOICE) >= 2 and KOKORO_VOICE[:2] in ("af", "am", "bf", "bm") else ""

# ── Runtime detection ────────────────────────────────────────────────────────
# Detect GPU; fall back to CPU if unavailable.
try:
    _r = subprocess.run(["nvidia-smi"], capture_output=True, timeout=5)
    has_gpu = _r.returncode == 0
except Exception:
    has_gpu = False

print(f"GPU available : {has_gpu}")
print(f"Voice         : {KOKORO_VOICE}")
print(f"Speed         : {KOKORO_SPEED}")
print('✅ Configuration set.')

## Step 4 — Start the API server

In [ ]:
import os, subprocess, time, urllib.request, json

# Stop any previously running server instance before starting a new one
if "server_proc" in dir() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except Exception:
        server_proc.kill()

# Build environment for the server process
server_env = os.environ.copy()
server_env.update({
    "KOKORO_VOICE":     KOKORO_VOICE,
    "KOKORO_SPEED":     KOKORO_SPEED,
    "KOKORO_PORT":      KOKORO_PORT,
    "KOKORO_API_KEY":   KOKORO_API_KEY,
    "KOKORO_LANG_CODE": KOKORO_LANG_CODE,
    "HF_HOME":          "/content/kokoro_cache",
})

os.makedirs("/content/kokoro_cache", exist_ok=True)

log_file = open("/content/kokoro_server.log", "w")
server_proc = subprocess.Popen(
    ["python", "api_server.py"],
    cwd="/content",
    env=server_env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# Wait for the health endpoint to respond (model download + load can take a while)
health_url = f"http://127.0.0.1:{KOKORO_PORT}/health"
print(f"Starting server (pid {server_proc.pid}) — waiting for model to load…")
print("(The model will be downloaded from HuggingFace on first run — this may take a minute.)")

ready = False
for attempt in range(120):  # up to ~2 minutes
    time.sleep(2)
    try:
        with urllib.request.urlopen(health_url, timeout=3) as r:
            data = json.loads(r.read())
            print(f"\n✅ Server ready — engine: {data.get('engine')}")
            ready = True
            break
    except Exception:
        if attempt % 5 == 0:
            print(".", end="", flush=True)

if not ready:
    print("\n❌ Server did not become ready in time. Check logs:")
    with open("/content/kokoro_server.log") as _f:
        print(_f.read()[-3000:])

## Step 5 — Generate speech

The cell below uses a built-in sample text so you can try the API right away.  
To use your own text instead, uncomment and run the **Enter your own text** cell, then re-run the **Set up API connection** cell.

### Use built-in sample text

In [ ]:
speech_text = (
    "Hello! This is a text-to-speech demo powered by Kokoro and docker-kokoro. "
    "The Kokoro model can synthesize natural-sounding speech in multiple voices "
    "and accents, and this API is compatible with the OpenAI audio speech endpoint."
)
print(f"Text ({len(speech_text)} chars):")
print(speech_text)
print('✅ Text ready.')

### (Optional) Enter your own text instead

Run this cell manually to set your own text. It will override the sample above.

In [ ]:
# NOTE: Run this cell manually only — it is intentionally skipped during 'Run all'.
# Uncomment the line below and run this cell to use your own text.

# speech_text = "Enter your text here."
# print(f"Text ({len(speech_text)} chars):")
# print(speech_text)

### Set up API connection

Run this cell once after choosing text above.

In [ ]:
import os, requests

assert speech_text and speech_text.strip(), \
    "No text set — run the sample text cell or custom text cell above first."

API_BASE = f"http://127.0.0.1:{KOKORO_PORT}"

req_headers = {"Content-Type": "application/json"}
if KOKORO_API_KEY:
    req_headers["Authorization"] = f"Bearer {KOKORO_API_KEY}"

print(f"API base : {API_BASE}")
print(f"Voice    : {KOKORO_VOICE}")
print('✅ API connection ready.')

### Generate speech — WAV (batch)

In [ ]:
import IPython.display as ipd

response = requests.post(
    f"{API_BASE}/v1/audio/speech",
    headers=req_headers,
    json={
        "model": "tts-1",
        "input": speech_text,
        "voice": KOKORO_VOICE,
        "response_format": "wav",
        "speed": float(KOKORO_SPEED),
    },
)

response.raise_for_status()
audio_bytes = response.content
print(f"✅ Received {len(audio_bytes):,} bytes of WAV audio")

# Save and play
wav_path = "/content/kokoro_output.wav"
with open(wav_path, "wb") as f:
    f.write(audio_bytes)

print("🔊 Playing audio:")
ipd.display(ipd.Audio(wav_path))

### Generate speech — WAV (streaming)

With `stream=true` the server yields audio chunks as each sentence is synthesized, reducing time-to-first-audio for long texts.

In [ ]:
import IPython.display as ipd

from requests.exceptions import ChunkedEncodingError

chunks = []
with requests.post(
    f"{API_BASE}/v1/audio/speech",
    headers=req_headers,
    json={
        "model": "tts-1",
        "input": speech_text,
        "voice": KOKORO_VOICE,
        "response_format": "wav",
        "speed": float(KOKORO_SPEED),
        "stream": True,
    },
    stream=True,
) as resp:
    resp.raise_for_status()
    try:
        for chunk in resp.iter_content(chunk_size=4096):
            if chunk:
                chunks.append(chunk)
    except ChunkedEncodingError:
        pass  # All audio data received; server closed connection after last chunk

stream_bytes = b"".join(chunks)
print(f"✅ Received {len(stream_bytes):,} bytes via streaming")

stream_path = "/content/kokoro_stream.wav"
with open(stream_path, "wb") as f:
    f.write(stream_bytes)

print("🔊 Playing streamed audio:")
ipd.display(ipd.Audio(stream_path))

### List all available voices

In [ ]:
response = requests.get(f"{API_BASE}/v1/voices", headers=req_headers)
response.raise_for_status()
voices = response.json()["voices"]
print(f"{len(voices)} voices available:\n")
for v in voices:
    print(f"  {v['id']:<18}  {v['description']}")
print('✅ Done.')

## View server logs

In [ ]:
!tail -40 /content/kokoro_server.log

## Stop the server

In [ ]:
if "server_proc" in dir() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except Exception:
        server_proc.kill()
    print("✅ Server stopped.")
else:
    print("ℹ️  Server is not running.")